In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg, count, round, datediff, when, to_timestamp

spark = SparkSession.builder.appName("olist_gold").getOrCreate()

df_orders = spark.read.parquet('../data/silver/orders')
df_order_items = spark.read.parquet('../data/silver/order_items')
df_order_payments = spark.read.parquet('../data/silver/order_payments')
df_order_reviews = spark.read.parquet('../data/silver/order_reviews')
df_customers = spark.read.parquet('../data/silver/customers')
df_products = spark.read.parquet('../data/silver/products')
df_sellers = spark.read.parquet('../data/silver/sellers')
df_category = spark.read.parquet('../data/silver/category')
df_geolocation = spark.read.parquet('../data/silver/geolocation')

In [18]:
df_gold = df_orders \
    .join(df_customers, "customer_id") \
    .join(df_order_items, "order_id") \
    .join(df_products, "product_id") \
    .join(df_sellers, "seller_id") \
    .join(df_category, "product_category_name", "left") \
    #.join(df_order_reviews, "order_id", "left")
    #.join(df_order_payments, "order_id", "left") \
    


In [19]:
print(f"Nombre de lignes : {df_gold.count()}")
print(f"Nombre de colonnes : {len(df_gold.columns)}")

cols_importantes = ["order_id", "customer_id", "product_id", "seller_id", "price"]
for c in cols_importantes:
    nulls = df_gold.filter(col(c).isNull()).count()
    print(f"  {c} : {nulls} nulls")

df_gold.select("order_id", "customer_id", "price", "product_category_name_english").show(5)

Nombre de lignes : 112598
Nombre de colonnes : 30
  order_id : 0 nulls
  customer_id : 0 nulls
  product_id : 0 nulls
  seller_id : 0 nulls
  price : 0 nulls
+--------------------+--------------------+-----+-----------------------------+
|            order_id|         customer_id|price|product_category_name_english|
+--------------------+--------------------+-----+-----------------------------+
|00048cc3ae777c65d...|816cbea969fe5b689...| 21.9|                   housewares|
|000e63d38ae8c00bb...|98884e672c5ba85f4...| 47.9|               bed_bath_table|
|00119ff934e539cf2...|7dd2e283f47deac85...|219.9|               sports_leisure|
|001427c0ec99cf8af...|eab9c552374be06ed...| 59.9|              furniture_decor|
|0017afd5076e074a4...|8085a9af46f619bc2...|809.1|         computers_accesso...|
+--------------------+--------------------+-----+-----------------------------+
only showing top 5 rows


In [22]:
from pyspark.sql.functions import sum, avg, round, count, desc, month, year

df_items_base = df_order_items \
    .join(df_orders, "order_id") \
    .join(df_products, "product_id") \
    .join(df_sellers, "seller_id") \
    .join(df_category, "product_category_name", "left")

# Chiffre d'affaires total
df_items_base.agg(round(sum("price"), 2).alias("ca_total")).show()

# Panier moyen par commande
df_items_base.groupBy("order_id").agg(sum("price").alias("total")) \
    .agg(round(avg("total"), 2).alias("panier_moyen")).show()

# Top 10 catégories
df_items_base.groupBy("product_category_name_english") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

# Top 10 vendeurs
df_items_base.groupBy("seller_id") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

+-------------+
|     ca_total|
+-------------+
|1.358764606E7|
+-------------+

+------------+
|panier_moyen|
+------------+
|      137.75|
+------------+

+-----------------------------+----------+
|product_category_name_english|        ca|
+-----------------------------+----------+
|                health_beauty|1258549.54|
|                watches_gifts|1204938.79|
|               bed_bath_table|1036574.08|
|               sports_leisure| 987863.98|
|         computers_accesso...| 911771.45|
|              furniture_decor| 728555.69|
|                   cool_stuff| 635202.86|
|                   housewares| 632050.16|
|                         auto| 592694.12|
|                 garden_tools| 484974.47|
+-----------------------------+----------+
only showing top 10 rows
+--------------------+---------+
|           seller_id|       ca|
+--------------------+---------+
|4869f7a5dfa277a7d...|229472.63|
|53243585a1d6dc264...|222776.05|
|4a3ca9315b744ce9f...|200472.92|
|fa1c13f2614d7b5c4

In [24]:
from pyspark.sql.functions import month, year, trunc

df_items_base \
    .withColumn("mois", trunc("order_purchase_timestamp", "MM")) \
    .groupBy("mois") \
    .agg(round(sum("price"), 2).alias("ca_mensuel")) \
    .orderBy("mois") \
    .show(24)

+----------+----------+
|      mois|ca_mensuel|
+----------+----------+
|2016-09-01|    267.36|
|2016-10-01|  48920.72|
|2016-12-01|      10.9|
|2017-01-01| 120124.22|
|2017-02-01| 246912.43|
|2017-03-01|  374255.3|
|2017-04-01| 359927.23|
|2017-05-01| 505781.16|
|2017-06-01| 432896.71|
|2017-07-01| 496431.58|
|2017-08-01| 573499.89|
|2017-09-01| 624222.69|
|2017-10-01| 664219.43|
|2017-11-01|1010271.37|
|2017-12-01| 743914.17|
|2018-01-01| 950030.36|
|2018-02-01| 844178.71|
|2018-03-01| 983153.54|
|2018-04-01| 996647.75|
|2018-05-01| 996517.68|
|2018-06-01| 865124.31|
|2018-07-01| 895507.22|
|2018-08-01| 854686.33|
|2018-09-01|     145.0|
+----------+----------+



In [25]:
# Délai moyen de livraison et taux de retard
from pyspark.sql.functions import datediff, when

df_livraison = df_orders.filter(col("order_delivered_customer_date").isNotNull())

# Délai moyen de livraison
df_livraison \
    .withColumn("delai", datediff("order_delivered_customer_date", "order_purchase_timestamp")) \
    .agg(round(avg("delai"), 1).alias("delai_moyen_jours")) \
    .show()

# Taux de retard
df_livraison \
    .withColumn("en_retard", when(
        col("order_delivered_customer_date") > col("order_estimated_delivery_date"), 1
    ).otherwise(0)) \
    .agg(
        round(avg("en_retard") * 100, 2).alias("taux_retard_pct")
    ) \
    .show()

+-----------------+
|delai_moyen_jours|
+-----------------+
|             12.5|
+-----------------+

+---------------+
|taux_retard_pct|
+---------------+
|           8.11|
+---------------+



In [26]:
# Note moyenne client
df_order_reviews.agg(round(avg("review_score"), 2).alias("note_moyenne")).show()

# Impact des retards sur la satisfaction
df_orders \
    .filter(col("order_delivered_customer_date").isNotNull()) \
    .withColumn("en_retard", when(
        col("order_delivered_customer_date") > col("order_estimated_delivery_date"), "retard"
    ).otherwise("a_temps")) \
    .join(df_order_reviews, "order_id", "left") \
    .groupBy("en_retard") \
    .agg(round(avg("review_score"), 2).alias("note_moyenne")) \
    .show()

+------------+
|note_moyenne|
+------------+
|        4.09|
+------------+

+---------+------------+
|en_retard|note_moyenne|
+---------+------------+
|   retard|        2.57|
|  a_temps|        4.29|
+---------+------------+



In [29]:
# Taux de commandes par statut
df_orders \
    .groupBy("order_status") \
    .agg(count("order_id").alias("nb_commandes")) \
    .orderBy(desc("nb_commandes")) \
    .show()

# CA par état brésilien
df_items_base \
    .join(df_customers, "customer_id") \
    .groupBy("customer_state") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

# Note moyenne par catégorie
df_items_base \
    .join(df_order_reviews, "order_id", "left") \
    .groupBy("product_category_name_english") \
    .agg(round(avg("review_score"), 2).alias("note_moyenne")) \
    .orderBy(desc("note_moyenne")) \
    .show(10)

+------------+------------+
|order_status|nb_commandes|
+------------+------------+
|   delivered|       96453|
|     shipped|        1107|
|    canceled|         625|
| unavailable|         609|
|    invoiced|         314|
|  processing|         301|
|     created|           5|
|    approved|           2|
+------------+------------+

+--------------+----------+
|customer_state|        ca|
+--------------+----------+
|            SP|5200592.28|
|            RJ|1823992.68|
|            MG|1584380.65|
|            RS| 750147.02|
|            PR| 683083.76|
|            SC| 520102.84|
|            BA| 511349.99|
|            DF| 302603.94|
|            GO| 294591.95|
|            ES| 275037.31|
+--------------+----------+
only showing top 10 rows
+-----------------------------+------------+
|product_category_name_english|note_moyenne|
+-----------------------------+------------+
|            cds_dvds_musicals|        4.64|
|         fashion_childrens...|         4.5|
|         books_gener

In [31]:
df_items_base \
    .join(df_customers, "customer_id") \
    .join(df_geolocation, 
          df_customers.customer_zip_code_prefix == df_geolocation.geolocation_zip_code_prefix,
          "left") \
    .groupBy("customer_state", "geolocation_lat", "geolocation_lng") \
    .agg(
        round(sum("price"), 2).alias("ca"),
        count("order_id").alias("nb_commandes")
    ) \
    .orderBy(desc("ca")) \
    .show(10)

+--------------+-------------------+-------------------+--------+------------+
|customer_state|    geolocation_lat|    geolocation_lng|      ca|nb_commandes|
+--------------+-------------------+-------------------+--------+------------+
|            DF|               NULL|               NULL|24852.67|         189|
|            RJ|-23.011335372924805| -43.45025634765625|22154.89|         154|
|            MG| -20.15825080871582| -44.88243865966797|18061.03|          34|
|            RJ| -22.90581703186035| -43.10698699951172|17727.17|         145|
|            RJ| -22.87187385559082| -42.26921844482422|17199.19|          89|
|            RJ|-22.911270141601562| -43.10515213012695|17100.69|         134|
|            RJ|  -23.0092716217041| -43.42940902709961|17067.71|         150|
|            RJ| -22.97295570373535|-43.397064208984375|16875.18|         123|
|            ES| -20.37580680847168| -40.30986022949219|16536.32|          82|
|            SP| -23.15171241760254| -46.97128677368

In [ ]:
df_items_base \
    .join(df_customers.select("customer_id", "customer_state"), "customer_id") \
    .groupBy("customer_state") \
    .agg(round(sum("price"), 2).alias("ca")) \
    .orderBy(desc("ca")) \
    .show(10)

+--------------+----------+
|customer_state|        ca|
+--------------+----------+
|            SP|5200592.28|
|            RJ|1823992.68|
|            MG|1584380.65|
|            RS| 750147.02|
|            PR| 683083.76|
|            SC| 520102.84|
|            BA| 511349.99|
|            DF| 302603.94|
|            GO| 294591.95|
|            ES| 275037.31|
+--------------+----------+
only showing top 10 rows


26/06/18 09:51:44 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 901427 ms exceeds timeout 120000 ms
26/06/18 09:51:44 WARN SparkContext: Killing executors is not supported by current scheduler.
26/06/18 09:51:48 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$